# Entrenamiento y evaluación en Colab

Notebook principal nuevo del proyecto.

In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

print("Imports OK — select one experiment cell below and run it.")

## Experiment: supervised

Fully supervised baseline — no unlabeled data used.  
Temporal branch: **OFF** (`use_semi=False`, `use_temp_consistency=False`).

In [ ]:
# =========================
# supervised
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/supervised"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only — no unlabeled data
cfg["seed"]                 = 0
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5
cfg["run_ruler_eval"] = True

print(summarize_config(cfg))

train_tf = get_supervised_train_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=None,
    temporal_unlab_ds=None,
)

artifacts = run_training(cfg, loaders)

results = evaluate_checkpoint(
    cfg,
    artifacts['model'],
    loaders,
    artifacts['best_path'],
    artifacts['history'],
)
print(results)

## Experiment: semi_std_matched_r3

Random matched control for `semi_r3`.  
Pool: `unlabeling_std_matched_r3/images` — same per-video count as r=3, uniform random selection, no temporal constraint.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_std_matched_r3
# =========================
cfg = get_default_config()

# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_std_matched_r3"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: random control matched to r=3 per-video counts
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

artifacts = run_training(cfg, loaders)

results = evaluate_checkpoint(
    cfg,
    artifacts['model'],
    loaders,
    artifacts['best_path'],
    artifacts['history'],
)
print(results)

## Experiment: semi_std_matched_r10

Random matched control for `semi_r10`.  
Pool: `unlabeling_std_matched_r10/images` — same per-video count as r=10, uniform random selection, no temporal constraint.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_std_matched_r10
# =========================
cfg = get_default_config()

# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_std_matched_r10"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: random control matched to r=10 per-video counts
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

artifacts = run_training(cfg, loaders)

results = evaluate_checkpoint(
    cfg,
    artifacts['model'],
    loaders,
    artifacts['best_path'],
    artifacts['history'],
)
print(results)

## Experiment: semi_r3

Temporal-neighbor semi-supervised run with r=3.  
Pool: `unlabeling_r3_max0/images`.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_r3
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_r3"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: temporal-neighbor r=3
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

artifacts = run_training(cfg, loaders)

results = evaluate_checkpoint(
    cfg,
    artifacts['model'],
    loaders,
    artifacts['best_path'],
    artifacts['history'],
)
print(results)

## Experiment: semi_r10

Temporal-neighbor semi-supervised run with r=10.  
Pool: `unlabeling_r10_max0/images`.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_r10
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_r10"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: temporal-neighbor r=10
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

artifacts = run_training(cfg, loaders)

results = evaluate_checkpoint(
    cfg,
    artifacts['model'],
    loaders,
    artifacts['best_path'],
    artifacts['history'],
)
print(results)